---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-37: LangChain <b>Retrievers</b></h1>

# Learning agenda of this notebook

> **The idea of this lecture:** a vector store (Lec-36) can find *similar* text, but in Example 6 of Lec-36 it did not find the *right* text for "Who are the authors of the paper?". A **retriever** is any component that takes a question and returns a list of Documents. Some search a vector store, some search the web, some rewrite the question, some re-score the results, and some do not use embeddings at all. In production, several are combined.

1. **What is a Retriever?**
2. **Setup: Two Stores from Lec-36**
3. **Retrievers on a Vector Store**
    - Example 1: `as_retriever()`: similarity search
    - Example 2: Only good matches: a score threshold
    - Example 3: Relevant but different: MMR
4. **Retrievers without a Vector Store**
    - Example 4: `WikipediaRetriever`
5. **Better Questions, Better Ranking**
    - Example 5: `MultiQueryRetriever`: an LLM asks the question in several ways
    - Example 6: Reranking: fetch many, keep the best
6. **Keyword Search and Hybrid Search**
    - Example 7: `BM25Retriever` on a transcript, then BM25 + vectors
7. **Summary**

> **Note:** Example 4 needs an internet connection and Example 5 uses the free `openai/gpt-oss-120b` model on Groq. Everything else runs locally; Example 6 downloads a small reranking model (about 90 MB) the first time.

# <span style='background :lightgreen' >1. What is a Retriever?</span>

In the RAG app of Lec-30, the retriever is the first step for every question: it decides which chunks the model will read.

<div style="text-align:center;">
    <img src="../images/langchain-new-03.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

A retriever has one job: **a question in, a list of `Document`s out**. It is a Runnable (Lec-33), so it has `invoke()` and `batch()` and can be joined with `|`. Unlike a vector store, it has no `add_documents()`: it only finds.

<div style="text-align:center;">
    <img src="../images/langchain-new-28.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

**More retrievers you will meet** (not in this lecture):

| Retriever | Idea | Package |
|---|---|---|
| `ParentDocumentRetriever` | search small chunks, but return the larger chunk (or page) they come from | `langchain-classic` |
| `SelfQueryRetriever` | an LLM turns "papers after 2020 about vision" into a search **plus a metadata filter** | `langchain-classic` |
| `MultiVectorRetriever` | store several vectors per Document (summary, questions, text) | `langchain-classic` |
| `TimeWeightedVectorStoreRetriever` | prefer recently used Documents | `langchain-classic` |
| `TavilySearchAPIRetriever` and other search APIs | search the live web | `langchain-tavily` and others |

> **Note:** in LangChain 1.x a retriever is called with `invoke()`. The old method `get_relevant_documents()` was removed. Many advanced retrievers now live in the `langchain-classic` package.

# <span style='background :lightgreen' >2. Setup: Two Stores from Lec-36</span>

We rebuild the two stores of Lec-36 in memory (about two seconds), so that this notebook runs on its own:
- `facts`: the 14 short facts (Lec-36, Examples 1 to 5), in Chroma with the cosine distance;
- `paper`: the 60 chunks of `attention.pdf` (Lec-36, Example 6).

In [1]:
import transformers                                                  # the library under sentence-transformers
from transformers import AutoTokenizer                               # to size chunks in MiniLM tokens (Lec-35)
from langchain_huggingface import HuggingFaceEmbeddings              # the free local embedding model
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

transformers.logging.set_verbosity_error()                           # hide library warnings
transformers.logging.disable_progress_bar()                          # hide the "Loading weights" progress bar
MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=MODEL)

# 1. the 14 facts of Lec-36
docs = [
    Document("Islamabad is the capital of Pakistan", metadata={"category": "Geography", "country": "Pakistan"}),
    Document("Paris is the capital of France", metadata={"category": "Geography", "country": "France"}),
    Document("Rome is the capital of Italy", metadata={"category": "Geography", "country": "Italy"}),
    Document("LangChain makes it easy to work with LLMs.", metadata={"category": "AI Tools"}),
    Document("LangChain is used to build LLM based applications.", metadata={"category": "AI Tools"}),
    Document("Chroma is used to store and search document embeddings.", metadata={"category": "Vector Database"}),
    Document("Embeddings are vector representations of text.", metadata={"category": "AI Concepts"}),
    Document("MMR helps you get diverse results when doing similarity search.", metadata={"category": "Search"}),
    Document("LangChain supports Chroma, FAISS, Pinecone, and more.", metadata={"category": "AI Tools"}),
    Document("Climate change is causing glaciers to melt rapidly in the Arctic region.", metadata={"category": "Environment", "region": "Arctic"}),
    Document("Glaciers in the Arctic are melting at an alarming rate due to rising temperatures.", metadata={"category": "Environment", "region": "Arctic"}),
    Document("Deforestation in the Amazon is accelerating global climate change.", metadata={"category": "Environment", "region": "Amazon"}),
    Document("Climate change is increasing the frequency of wildfires in California.", metadata={"category": "Environment", "region": "California"}),
    Document("Rising sea levels due to climate change threaten coastal cities like Mumbai and New York.", metadata={"category": "Environment", "region": "Global"}),
]
facts = Chroma.from_documents(docs, embeddings, ids=[f"doc-{i:02d}" for i in range(1, 15)],
                              collection_name="facts", collection_metadata={"hnsw:space": "cosine"})

# 2. the paper: load, split to 200 MiniLM tokens, store
pages = PyPDFLoader("../data/attention.pdf").load()
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    AutoTokenizer.from_pretrained(MODEL), chunk_size=200, chunk_overlap=20)
chunks = splitter.split_documents(pages)
paper = InMemoryVectorStore.from_documents(chunks, embeddings)

print(len(docs), "facts and", len(chunks), "chunks of the paper are stored")

def show(results, width=95):
    """Print each Document: page number if it has one, then the start of its text."""
    for i, doc in enumerate(results, 1):
        page = f"page {doc.metadata['page_label']:>2} | " if "page_label" in doc.metadata else ""
        print(f"   {i}. {page}{doc.page_content[:width].replace(chr(10), ' ')}")

14 facts and 60 chunks of the paper are stored


# <span style='background :lightgreen' >3. Retrievers on a Vector Store</span>

## <span style='color:#7e22ce'>Example 1: <code>as_retriever()</code>: similarity search</span>

`as_retriever()` turns any vector store into a retriever. Two arguments decide how it searches:
- `search_type`: `"similarity"` (default), `"similarity_score_threshold"` or `"mmr"`;
- `search_kwargs`: the settings of that search, for example `{"k": 3}`.

In [2]:
retriever = facts.as_retriever(search_type="similarity", search_kwargs={"k": 3})   # the 3 nearest Documents
print(type(retriever).__name__)                                                    # VectorStoreRetriever

show(retriever.invoke("What is the capital of Pakistan?"))       # one question -> a list of Documents

print()
answers = retriever.batch(["Which city is the capital of France?", "What are embeddings?"])   # a Runnable: batch works
for result in answers:
    show(result[:1])                                             # the best Document for each question

VectorStoreRetriever
   1. Islamabad is the capital of Pakistan
   2. Rome is the capital of Italy
   3. Paris is the capital of France

   1. Paris is the capital of France
   1. Embeddings are vector representations of text.


The retriever does exactly what `similarity_search()` did in Lec-36, but behind the common retriever interface, so it can be replaced by any other retriever in this lecture without changing the rest of the app.

The answer to "What is the capital of Pakistan?" is the first Document, but `k=3` always returns three Documents, even when the other two (Rome, Paris) are useless.

## <span style='color:#7e22ce'>Example 2: Only good matches: a score threshold</span>

`"similarity_score_threshold"` returns only the Documents whose **relevance score** (Lec-36, Example 2: higher is better, 0 to 1) is above a threshold. It can return fewer than `k` Documents, or none.

In [3]:
strict = facts.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.3, "k": 3},      # keep relevance >= 0.3, at most 3 Documents
)
show(strict.invoke("What is the capital of Pakistan?"))   # Islamabad 0.81, Rome 0.34; Paris 0.21 is dropped
print()
nothing = strict.invoke("How do I bake bread?")             # nothing in our facts is about bread
print("bread:", len(nothing), "Documents")                  # an empty list, not three wrong answers

No relevant docs were retrieved using the relevance score threshold 0.3


   1. Islamabad is the capital of Pakistan
   2. Rome is the capital of Italy

bread: 0 Documents


LangChain also prints a warning when nothing passes the threshold. A threshold lets the app say *"I could not find this in the documents"* instead of answering from unrelated text. The right value depends on the embedding model and the store's score (Lec-36, figure in Example 2), so test it on your own questions.

## <span style='color:#7e22ce'>Example 3: Relevant but different: MMR</span>

The top `k` results are often **near-copies** of each other: two facts about melting Arctic glaciers say the same thing twice. **Maximal Marginal Relevance (MMR)** picks Documents one at a time: each new one must be relevant to the question **and** different from the ones already picked.
- `fetch_k`: how many candidates to fetch first (by similarity);
- `k`: how many to return from them;
- `lambda_mult`: the balance, from `1.0` (only relevance, like similarity search) to `0.0` (only diversity).

In [4]:
question = "Tell me about melting glaciers"

print("similarity, k=3:")
show(facts.as_retriever(search_kwargs={"k": 3}).invoke(question))

print("\nMMR, k=3 from the 5 most similar (fetch_k=5):")
mmr = facts.as_retriever(search_type="mmr", search_kwargs={"k": 3, "fetch_k": 5, "lambda_mult": 0.5})
show(mmr.invoke(question))

print("\nMMR, k=3 from all 14 facts (fetch_k=14):")
mmr_all = facts.as_retriever(search_type="mmr", search_kwargs={"k": 3, "fetch_k": 14, "lambda_mult": 0.5})
show(mmr_all.invoke(question))

similarity, k=3:
   1. Glaciers in the Arctic are melting at an alarming rate due to rising temperatures.
   2. Climate change is causing glaciers to melt rapidly in the Arctic region.
   3. Rising sea levels due to climate change threaten coastal cities like Mumbai and New York.

MMR, k=3 from the 5 most similar (fetch_k=5):
   1. Glaciers in the Arctic are melting at an alarming rate due to rising temperatures.
   2. Rising sea levels due to climate change threaten coastal cities like Mumbai and New York.
   3. Deforestation in the Amazon is accelerating global climate change.

MMR, k=3 from all 14 facts (fetch_k=14):
   1. Glaciers in the Arctic are melting at an alarming rate due to rising temperatures.
   2. LangChain supports Chroma, FAISS, Pinecone, and more.
   3. Rome is the capital of Italy


- **Similarity** returns the two glacier facts, which say the same thing twice.
- **MMR with `fetch_k=5`** keeps one of them and uses the free place for another climate fact (the Amazon).
- **MMR with `fetch_k=14`** may choose from *all* facts, so it picks unrelated ones simply because they are "different".

`fetch_k` decides where MMR may look for variety: keep it small enough that all candidates are still relevant (often 2 to 5 times `k` in a large collection), and `lambda_mult` around 0.5 to 0.8.

# <span style='background :lightgreen' >4. Retrievers without a Vector Store</span>

## <span style='color:#7e22ce'>Example 4: <code>WikipediaRetriever</code></span>

A retriever does not need a vector store of our own. `WikipediaRetriever` sends the question to Wikipedia's search and returns the best articles as Documents. No embeddings, no loading, no splitting: useful for general knowledge, and always up to date.

In [5]:
import wikipedia                                                  # the library that WikipediaRetriever uses
from langchain_community.retrievers import WikipediaRetriever

# Wikipedia blocks the library's default User-Agent (HTTP 403/429), so we say who we are
wikipedia.set_user_agent("GenAI-Course-Lec37/1.0 (University of the Punjab; educational use)")

wiki = WikipediaRetriever(top_k_results=2, lang="en", doc_content_chars_max=2000)   # 2 articles, 2000 characters each
wiki_docs = wiki.invoke("Durand Line agreement")
for doc in wiki_docs:
    print(doc.metadata["title"], "|", doc.metadata["source"])     # the article title and its URL
    print("   ", doc.page_content[:200].replace("\n", " "), "...\n")

Durand Line | https://en.wikipedia.org/wiki/Durand_Line
    The Durand Line, also known as the Afghanistan–Pakistan International border, is a 2,640-kilometre (1,640 mi) international border between Afghanistan and Pakistan. The western end runs to the border  ...

Parachinar | https://en.wikipedia.org/wiki/Parachinar
    Parachinar (Pashto: پاړه چنار; Urdu: پارا چنار) is a city and the headquarters of the Kurram District in the province of Khyber Pakhtunkhwa, Pakistan. Parachinar, situated on the west of Peshawar, jut ...



> **Note:** Wikipedia allows only a few requests per minute from one program; run this cell once and reuse `wiki_docs`. Other web retrievers work the same way, for example search APIs such as Tavily. (`ArxivRetriever` for research papers currently breaks with the newest `arxiv` package, version 4.)

# <span style='background :lightgreen' >5. Better Questions, Better Ranking</span>

Back to the question that failed in Lec-36, Example 6. Plain similarity search returns three chunks of the **reference list**, not page 1 with the author names:

In [6]:
QUESTION = "Who are the authors of the paper?"
dense = paper.as_retriever(search_kwargs={"k": 3})          # plain similarity search on the paper
show(dense.invoke(QUESTION))

   1. page 11 | Information Processing Systems, (NIPS), 2016. [17] Łukasz Kaiser and Ilya Sutskever. Neural GPU
   2. page 12 | [25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building a large annota
   3. page 12 | Macherey, Maxim Krikun, Yuan Cao, Qin Gao, Klaus Macherey, et al. Google’s neural machine trans


## <span style='color:#7e22ce'>Example 5: <code>MultiQueryRetriever</code>: an LLM asks the question in several ways</span>

A question can be written in many ways, and each wording lands on a slightly different point in the vector space. `MultiQueryRetriever` asks an LLM for **three versions** of the question, runs the base retriever with each one, and returns the **union** of all results (without duplicates).

The `llm_chain` is a small chain from Lec-33: prompt, model, and a function that returns a `list[str]`. We use `with_structured_output()` so that the model returns a real list. (`MultiQueryRetriever.from_llm()` builds a chain that expects one question per line; our model sometimes writes the line breaks as the two characters `\n`, and then the retriever searches with one long, garbled question.)

In [7]:
import os, sys, logging                                       # logging: to see the questions the LLM writes
from dotenv import load_dotenv
from pydantic import BaseModel, Field                         # to describe the LLM's answer (Lec-30, Example 9)
from langchain_openai import ChatOpenAI                       # works with Groq's OpenAI-compatible API (Lec-30)
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.retrievers import MultiQueryRetriever

load_dotenv("../keys/.env", override=True)
GROQ_MODEL = ChatOpenAI(model="openai/gpt-oss-120b", base_url="https://api.groq.com/openai/v1",
                        api_key=os.getenv("GROQ_API_KEY"), max_retries=5)        # free

class Questions(BaseModel):
    questions: list[str] = Field(description="three different ways to ask the user's question")

rewrite_prompt = ChatPromptTemplate.from_template(
    "Write three different versions of this question, to search a document with a vector database. "
    "Question: {question}")
rewrite_chain = rewrite_prompt | GROQ_MODEL.with_structured_output(Questions) | (lambda out: out.questions)  # -> list[str]

log = logging.getLogger("langchain_classic.retrievers.multi_query")   # the retriever logs the questions it uses
log.setLevel(logging.INFO)
if not log.handlers:
    log.addHandler(logging.StreamHandler(sys.stdout))                   # print the log to the cell output

multi = MultiQueryRetriever(retriever=dense, llm_chain=rewrite_chain)   # wraps the plain retriever
results = multi.invoke(QUESTION)
print(f"\n{len(results)} different chunks:")
show(results)

Generated queries: ['Who authored the paper?', 'Can you list the authors of the paper?', 'Who wrote this paper?']

5 different chunks:
   1. page 11 | Information Processing Systems, (NIPS), 2016. [17] Łukasz Kaiser and Ilya Sutskever. Neural GPU
   2. page  1 | Provided proper attribution is provided, Google hereby grants permission to reproduce the table
   3. page 12 | Macherey, Maxim Krikun, Yuan Cao, Qin Gao, Klaus Macherey, et al. Google’s neural machine trans
   4. page 12 | [25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building a large annota
   5. page 10 | learning to align and translate. CoRR, abs/1409.0473, 2014. [3] Denny Britz, Anna Goldie, Minh-


In our run, the new wordings brought **page 1** into the results (the LLM writes different questions each time, so your list may differ). Multi-query search finds more (better **recall**), but it costs one LLM call per question and returns more Documents, which may not be in the best order. The next example fixes the order.

## <span style='color:#7e22ce'>Example 6: Reranking: fetch many, keep the best</span>

Our embedding model is a **bi-encoder**: it turns the question and each chunk into vectors **separately**, then compares the vectors. That is fast enough for millions of chunks, but the model never sees the question and the chunk together.

A **cross-encoder** reads the question **and** one chunk **together** and outputs one relevance score. It is much more accurate, but too slow to run on every chunk. So production systems use two stages:
1. **retrieve broadly**: the 20 nearest chunks with the fast bi-encoder;
2. **rerank narrowly**: score only those 20 with the cross-encoder, keep the best 3.

`ContextualCompressionRetriever` wraps a retriever and passes its results through a **compressor**; here the compressor is `CrossEncoderReranker`, with the free model `cross-encoder/ms-marco-MiniLM-L6-v2`, trained on Bing search questions.

In [8]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder           # runs a cross-encoder locally
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L6-v2")   # about 90 MB, once

rerank = ContextualCompressionRetriever(
    base_retriever=paper.as_retriever(search_kwargs={"k": 20}),   # stage 1: 20 candidates, fast
    base_compressor=CrossEncoderReranker(model=cross_encoder, top_n=3),   # stage 2: re-score them, keep 3
)
show(rerank.invoke(QUESTION))

   1. page  1 | Provided proper attribution is provided, Google hereby grants permission to reproduce the table
   2. page 12 | [25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building a large annota
   3. page 12 | [29] Slav Petrov, Leon Barrett, Romain Thibaux, and Dan Klein. Learning accurate, compact, and 


In [9]:
# the cross-encoder's scores for the 3 chunks plain search returned, and for the reranker's winner
candidates = dense.invoke(QUESTION) + rerank.invoke(QUESTION)[:1]
scores = cross_encoder.score([(QUESTION, doc.page_content) for doc in candidates])    # (question, chunk) pairs
for doc, score in zip(candidates, scores):
    print(f"   {score:6.2f}  page {doc.metadata['page_label']:>2} | {doc.page_content[:70].replace(chr(10), ' ')}")

    -9.38  page 11 | Information Processing Systems, (NIPS), 2016. [17] Łukasz Kaiser and I
    -7.35  page 12 | [25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini
    -7.78  page 12 | Macherey, Maxim Krikun, Yuan Cao, Qin Gao, Klaus Macherey, et al. Goog
    -7.12  page  1 | Provided proper attribution is provided, Google hereby grants permissi


The cross-encoder gives **page 1 the highest score**, so it moves from outside the top 3 to first place. The reference-list chunks score lower. A cross-encoder's score is not limited to 0..1 (here they are all negative): only the order matters.

<div style="text-align:center;">
    <img src="../images/langchain-new-29.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

> **Note:** a reranker can only reorder what stage 1 found. If the right chunk is not among the 20 candidates, no reranker can bring it back, so stage 1 should be broad. Cloud rerankers (Cohere, Jina, Voyage) work the same way.

# <span style='background :lightgreen' >6. Keyword Search and Hybrid Search</span>

## <span style='color:#7e22ce'>Example 7: <code>BM25Retriever</code> on a transcript, then BM25 + vectors</span>

All retrievers so far compare **meanings** with embeddings. **BM25** ("Best Matching 25", 1994) is the classic algorithm of search engines such as Elasticsearch and Lucene. It compares **words**:
- it scores a Document higher when it contains the **words of the question**;
- a word counts more when it is **rare** in the collection (it says more about which Document is meant), and less when it appears in every Document;
- repeating a word helps, but less and less, and long Documents are not favoured just for being long.

It needs **no embeddings and no model**: it is fast, cheap and easy to explain. We try it on a document full of **codes and numbers**, where embeddings are weak: an M.Phil transcript (personal details removed).

### Step 1: one Document per course
`PyPDFLoader` writes every table cell on its own line (Lec-35, Example 7). A few lines of code turn each course into one short, readable Document.

In [10]:
import re                                                             # regular expressions, to find the course codes

text = PyPDFLoader("../data/arif_mphil_transcript_redacted.pdf").load()[0].page_content
lines = [line.strip() for line in text.splitlines() if line.strip()]  # one table cell per line, no empty lines
print(lines[16:25], "...")                                            # how a course looks in the extracted text

courses, semester, i = [], None, 0
while i < len(lines):
    if re.fullmatch(r"Semester \d", lines[i]):                        # "Semester 1" ... "Semester 4"
        semester = lines[i]
    if re.fullmatch(r"CS-\d{3}", lines[i]):                           # a course code starts a row, e.g. "CS-565"
        j = i + 1
        while not lines[j].isdigit():                                 # the title can span several lines
            j += 1
        title = " ".join(lines[i + 1:j])
        credits, marks, grade = lines[j], lines[j + 1], lines[j + 2]  # the next three cells of the row
        courses.append(Document(
            f"{lines[i]} {title}: {credits} credit hours, {marks} marks, grade {grade} ({semester}).",
            metadata={"code": lines[i], "semester": semester, "grade": grade}))
        i = j + 3
        continue
    i += 1

print(f"\n{len(courses)} courses:")
for doc in courses:
    print("  ", doc.page_content)

['Marks', 'Obtain', 'Grade', 'CS-541', 'Advance Software', 'Development', '3', '73', 'B'] ...

10 courses:
   CS-541 Advance Software Development: 3 credit hours, 73 marks, grade B (Semester 1).
   CS-510 Analysis and Design of Algorithms: 3 credit hours, 88 marks, grade A+ (Semester 1).
   CS-530 Advance Web Semantics: 3 credit hours, 85 marks, grade A+ (Semester 1).
   CS-520 Advance Computer Architecture: 3 credit hours, 90 marks, grade A+ (Semester 2).
   CS-525 Advance Operating Systems: 3 credit hours, 89 marks, grade A+ (Semester 2).
   CS-500 Digital Signal Processing: 3 credit hours, 90 marks, grade A+ (Semester 2).
   CS-515 Theory of Computation: 3 credit hours, 83 marks, grade A (Semester 2).
   CS-561 Expert Systems: 3 credit hours, 86 marks, grade A+ (Semester 3).
   CS-565 Computer Vision: 3 credit hours, 83 marks, grade A (Semester 3).
   CS-691 Thesis: 6 credit hours, 85 marks, grade A+ (Semester 4).


### Step 2: BM25 and dense search on the same Documents
BM25 splits text into words with a `preprocess_func`. The default only splits at spaces and keeps capital letters, so "CS-565," and "cs-565" would be different words. We give it a simple tokenizer: lower case, letters and digits only.

In [11]:
from langchain_community.retrievers import BM25Retriever     # needs the rank-bm25 package

def words(text: str) -> list[str]:
    """Lower-case words made of letters and digits: 'What grade in CS-565?' -> ['what', 'grade', 'in', 'cs', '565']."""
    return re.findall(r"\w+", text.lower())

bm25 = BM25Retriever.from_documents(courses, k=2, preprocess_func=words)          # builds the word index
course_dense = InMemoryVectorStore.from_documents(courses, embeddings).as_retriever(search_kwargs={"k": 2})

print(words("What grade did I get in CS-565?"))

# how much each word is worth (IDF): rare words count much more than words found in every course
idf = bm25.vectorizer.idf                                     # the BM25 index built by the rank-bm25 library
for w in ["565", "vision", "computer", "grade", "cs"]:
    print(f"   {w:9} {idf[w]:.3f}")

['what', 'grade', 'did', 'i', 'get', 'in', 'cs', '565']
   565       1.846
   vision    1.846
   computer  1.224
   grade     0.248
   cs        0.248


"565" appears in one course only, so it is worth a lot; "grade" and "cs" appear in every course, so they are worth almost nothing (BM25 even gives them a small fixed floor). Now three questions for both retrievers:

In [12]:
def compare(question):
    """Print the best course found by dense (embedding) search and by BM25."""
    print("Q:", question)
    print("   dense:", course_dense.invoke(question)[0].page_content[:48])
    print("   BM25 :", bm25.invoke(question)[0].page_content[:48])
    print()

compare("What grade did I get in CS-565?")                # an exact course code
compare("CS-510 marks")                                   # an exact course code
compare("Which course was about images and cameras?")     # a meaning, with no matching words

Q: What grade did I get in CS-565?
   dense: CS-530 Advance Web Semantics: 3 credit hours, 85
   BM25 : CS-565 Computer Vision: 3 credit hours, 83 marks

Q: CS-510 marks
   dense: CS-520 Advance Computer Architecture: 3 credit h
   BM25 : CS-510 Analysis and Design of Algorithms: 3 cred

Q: Which course was about images and cameras?
   dense: CS-565 Computer Vision: 3 credit hours, 83 marks
   BM25 : CS-510 Analysis and Design of Algorithms: 3 cred



- **Codes and numbers:** BM25 finds CS-565 and CS-510 at once, because "565" and "510" are rare words that match exactly. For the embedding model, "CS-565", "CS-530" and "CS-520" are almost the same text, so dense search picks the wrong course first.
- **Meaning:** "images and cameras" shares no word with "Computer Vision", so BM25 cannot find it; the embedding model knows that the two mean nearly the same thing.

### Step 3: hybrid search, the best of both
`EnsembleRetriever` runs both retrievers and merges their two ranked lists with **Reciprocal Rank Fusion (RRF)**: each Document gets `weight / (rank + 60)` from every list it appears in, and the totals decide the final order. Only ranks are used, so BM25 scores and cosine scores never have to be compared.

In [13]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid = EnsembleRetriever(retrievers=[bm25, course_dense], weights=[0.5, 0.5])   # equal say for both lists

for question in ["What grade did I get in CS-565?", "CS-510 marks", "Which course was about images and cameras?"]:
    print("Q:", question)
    show(hybrid.invoke(question)[:2], width=60)                                   # the top 2 after fusion

Q: What grade did I get in CS-565?
   1. CS-565 Computer Vision: 3 credit hours, 83 marks, grade A (S
   2. CS-530 Advance Web Semantics: 3 credit hours, 85 marks, grad
Q: CS-510 marks
   1. CS-510 Analysis and Design of Algorithms: 3 credit hours, 88
   2. CS-520 Advance Computer Architecture: 3 credit hours, 90 mar
Q: Which course was about images and cameras?
   1. CS-510 Analysis and Design of Algorithms: 3 credit hours, 88
   2. CS-565 Computer Vision: 3 credit hours, 83 marks, grade A (S


Hybrid search puts the right course in the top 2 for **all three** questions. This is why hybrid search (BM25 + vectors) followed by a reranker (Example 6) is the most common retrieval setup in production RAG systems; many vector databases (Elasticsearch, Weaviate, Qdrant, Pinecone, MongoDB Atlas) offer it built in.

<div style="text-align:center;">
    <img src="../images/langchain-new-30.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

**How BM25 differs from every other retriever in this lecture:**

| | Dense retrievers (Examples 1 to 6) | BM25 (Example 7) |
|---|---|---|
| Compares | meanings (vectors from an embedding model) | words (exact matches) |
| Needs | an embedding model, a vector store | only the text: no model, no GPU |
| Good at | synonyms, paraphrases, other languages | codes, names, numbers, rare technical terms |
| Weak at | exact identifiers ("CS-565" vs "CS-530") | words that differ but mean the same ("images" vs "vision") |
| Explaining a result | hard: 384 numbers | easy: which words matched, and how rare they are |

# <span style='background :lightgreen' >7. Summary</span>

| Ex | Retriever | Idea | Package |
|---|---|---|---|
| 1 | `as_retriever(search_type="similarity")` | the k nearest chunks | `langchain-core` |
| 2 | `as_retriever(search_type="similarity_score_threshold")` | only chunks above a relevance score | `langchain-core` |
| 3 | `as_retriever(search_type="mmr")` | relevant **and** different | `langchain-core` |
| 4 | `WikipediaRetriever` | search an outside source | `langchain-community` |
| 5 | `MultiQueryRetriever` | an LLM rewrites the question several times | `langchain-classic` |
| 6 | `ContextualCompressionRetriever` + `CrossEncoderReranker` | fetch 20, rerank, keep 3 | `langchain-classic`, `langchain-community` |
| 7 | `BM25Retriever`, `EnsembleRetriever` | keyword search, and keywords + vectors | `langchain-community`, `langchain-classic` |

### Key takeaways
- A **retriever** takes a question and returns Documents; every retriever is a Runnable with `invoke()`.
- `as_retriever()` turns a vector store into a retriever: similarity, a score threshold, or MMR.
- Wrappers improve a retriever: **multi-query** finds more, a **reranker** puts the best first.
- **BM25** matches words, not meanings: strong on codes and names, blind to synonyms, and it needs no model.
- **Hybrid search + reranking** is the common production recipe.
- **Next lecture (Lec-38):** put a retriever, a prompt and a model in one chain, and compare it with RAG that uses no vectors at all.

### Check your understanding
1. What goes into a retriever and what comes out? Which method do you call?
2. Why can `"similarity_score_threshold"` return an empty list, and why is that useful?
3. What do `k`, `fetch_k` and `lambda_mult` control in MMR?
4. How does `MultiQueryRetriever` find more chunks than plain similarity search? What does it cost?
5. What is the difference between a bi-encoder and a cross-encoder? Why do we use both, in two stages?
6. Why can a reranker not fix a search whose candidates do not contain the right chunk?
7. Why does BM25 give "565" a much higher weight than "grade"?
8. Why did dense search confuse CS-565 with another course, and why did BM25 miss "images and cameras"?
9. How does Reciprocal Rank Fusion combine two ranked lists, and why does it not need the scores?